# Banking Intent Classifier: Preprocessing (branch `preprocessing`)

This notebook covers the **shared foundation** (data, split, label encoding) and **text preparation** (tokenizer, vocabulary, padding), plus the **data leakage checks**. It trains no model.

**Rule:** vocabulary, max length and label classes are fitted on the **train** split only. Validation and test are only transformed.

## Review checklist (for the reviewer)

- [ ] The notebook runs from top to bottom without errors (Run all).
- [ ] `TextPreprocessor.fit` is called on **train text only** (cell A2).
- [ ] The test set is loaded but not used to make any decision.
- [ ] The automatic shape and label checks pass (cell A3).
- [ ] The leakage checks in A4 run and the notes at the end are filled in.
- [ ] Names used by the model branch are clear: `clean_text`, `tokenize`, `TextPreprocessor`, `preprocessor`, `label_encoder`, `X_train / X_val / X_test`, `y_train / y_val / y_test`.

## 0. Setup

In [ ]:
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

---
# Part 1: Shared decisions

| Decision | Our choice |
|---|---|
| Task | Classify a customer banking query into one of 77 intents using its text |
| Text column / label column | `text` / `category` |
| Train / validation / test | Official PolyAI `train.csv` is split 80% train / 20% validation (stratified, seed 42). Official `test.csv` is the test set and stays untouched until the final evaluation |
| Related examples (author, customer, conversation) | The dataset has no such column, so nothing needs to be grouped. Stratification is therefore safe |
| Seed | `SEED = 42` everywhere |
| Label encoding | One `LabelEncoder` fitted on train labels, shared by all models |
| Text preparation | One shared preprocessing: lowercase, keep punctuation, keep stopwords |

### 1.1 Load the data and create train / validation / test

In [ ]:
train_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
test_url = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"

TEXT_COL = "text"
LABEL_COL = "category"

full_train_df = pd.read_csv(train_url)
test_df = pd.read_csv(test_url)

train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.2,
    stratify=full_train_df[LABEL_COL],
    random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Original train file:", full_train_df.shape)
print("Train      :", train_df.shape)
print("Validation :", val_df.shape)
print("Test       :", test_df.shape, "(untouched until the final evaluation)")

### 1.2 Document the split and class counts

In [ ]:
split_counts = pd.DataFrame({
    "train": train_df[LABEL_COL].value_counts(),
    "validation": val_df[LABEL_COL].value_counts(),
    "test": test_df[LABEL_COL].value_counts(),
}).fillna(0).astype(int)

summary = pd.DataFrame({
    "rows": split_counts.sum(),
    "classes": (split_counts > 0).sum(),
    "min per class": split_counts.min(),
    "max per class": split_counts.max(),
})
display(summary)

print("Smallest classes (train / validation / test):")
display(split_counts.sort_values("train").head(5))

### 1.3 Label encoding (shared by every model)

In [ ]:
label_encoder = LabelEncoder().fit(train_df[LABEL_COL])
CLASSES = label_encoder.classes_
N_CLASSES = len(CLASSES)

y_train = label_encoder.transform(train_df[LABEL_COL])
y_val = label_encoder.transform(val_df[LABEL_COL])
y_test = label_encoder.transform(test_df[LABEL_COL])

print("Number of classes:", N_CLASSES)
print("First 5 classes in the shared order:", list(CLASSES[:5]))

---
# Part 2: Prepare the text

### A1. Reusable preprocessing functions and class

In [ ]:
TOKEN_PATTERN = re.compile(r"[a-z0-9]+(?:'[a-z]+)?|[^\sa-z0-9]")
PAD_TOKEN, OOV_TOKEN = "<PAD>", "<OOV>"
PAD_ID, OOV_ID = 0, 1

def clean_text(text):
    """Minimal cleaning: lowercase, trim, collapse repeated whitespace."""
    text = str(text).lower().strip()
    return re.sub(r"\s+", " ", text)

def tokenize(text):
    """Split a text into word tokens and punctuation tokens."""
    return TOKEN_PATTERN.findall(clean_text(text))

class TextPreprocessor:
    """Text -> padded integer matrix.

    fit(texts)        learns the vocabulary and max length (call with TRAIN text only)
    transform(texts)  returns an int32 array of shape (n_texts, max_len_)
    """

    def __init__(self, max_vocab=5000, max_len=None, len_percentile=95):
        self.max_vocab = max_vocab
        self.max_len = max_len
        self.len_percentile = len_percentile

    def tokens(self, text):
        """The ONE place where text becomes tokens. Override this to change the preparation."""
        return tokenize(text)

    def fit(self, texts):
        tokenized = [self.tokens(t) for t in texts]
        counts = Counter(tok for toks in tokenized for tok in toks)
        kept = [tok for tok, _ in counts.most_common(self.max_vocab - 2)]
        self.itos = [PAD_TOKEN, OOV_TOKEN] + kept
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        if self.max_len is None:
            lengths = [len(toks) for toks in tokenized]
            self.max_len_ = int(np.percentile(lengths, self.len_percentile))
        else:
            self.max_len_ = int(self.max_len)
        return self

    @property
    def vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        return [self.stoi.get(tok, OOV_ID) for tok in self.tokens(text)]

    def decode(self, ids):
        return [self.itos[i] for i in ids if i != PAD_ID]

    def transform(self, texts):
        texts = list(texts)
        X = np.zeros((len(texts), self.max_len_), dtype="int32")
        for row, text in enumerate(texts):
            ids = self.encode(text)[: self.max_len_]
            X[row, : len(ids)] = ids
        return X

    def fit_transform(self, texts):
        return self.fit(texts).transform(texts)

### A2. 🔌 PREPROCESSING INTEGRATION POINT

This is the **only place** where the preprocessing is created and fitted. Track A hands this object to Track B.
To swap in a more advanced preprocessing later, replace the `preprocessor = ...` line with any object that has:
`.fit(texts)`, `.transform(texts)` returning an int array of shape `(n, max_len)`, plus `.vocab_size` and `.max_len_`.

In [ ]:
# 🔌 PREPROCESSING INTEGRATION POINT -----------------------------------------
preprocessor = TextPreprocessor(max_vocab=5000, len_percentile=95)
preprocessor.fit(train_df[TEXT_COL])          # TRAIN text only
# ----------------------------------------------------------------------------

X_train = preprocessor.transform(train_df[TEXT_COL])
X_val = preprocessor.transform(val_df[TEXT_COL])
X_test = preprocessor.transform(test_df[TEXT_COL])

train_lengths = np.array([len(preprocessor.tokens(t)) for t in train_df[TEXT_COL]])
print("Vocabulary size (with PAD and OOV):", preprocessor.vocab_size)
print("Max sequence length chosen:", preprocessor.max_len_)
print("Train token length: mean =", round(train_lengths.mean(), 1),
      "| 95th percentile =", int(np.percentile(train_lengths, 95)),
      "| max =", train_lengths.max())
print("Train texts that get truncated:",
      f"{(train_lengths > preprocessor.max_len_).mean():.1%}")
print("Shapes:", X_train.shape, X_val.shape, X_test.shape)

### A3. Inspect tokenized examples and check shapes and labels

In [ ]:
for i in train_df.sample(4, random_state=SEED).index:
    print("Text   :", train_df.loc[i, TEXT_COL])
    print("Tokens :", preprocessor.tokens(train_df.loc[i, TEXT_COL]))
    print("IDs    :", X_train[i].tolist())
    print("Decoded:", preprocessor.decode(X_train[i]))
    print("Label  :", y_train[i], "->", CLASSES[y_train[i]])
    print()

# Automatic checks
assert X_train.shape == (len(train_df), preprocessor.max_len_)
assert X_val.shape == (len(val_df), preprocessor.max_len_)
assert X_test.shape == (len(test_df), preprocessor.max_len_)
assert len(X_train) == len(y_train) == len(train_df)
assert (label_encoder.inverse_transform(y_train) == train_df[LABEL_COL].values).all()
assert (label_encoder.inverse_transform(y_val) == val_df[LABEL_COL].values).all()
assert X_train.max() < preprocessor.vocab_size and X_train.min() >= 0
print("All shape and label checks passed.")

### A4. Data leakage checks

Questions: are similar examples in train and validation/test? Do the same authors come back? Is anything identifying in the text?
Vocabulary and max length were fitted on train only, so validation/test words that are not in train become `<OOV>`.

In [ ]:
# 1) Out-of-vocabulary rate (should be higher on val/test than on train, because the vocabulary is train-only)
def oov_rate(texts):
    tokens = [tok for t in texts for tok in preprocessor.tokens(t)]
    return sum(tok not in preprocessor.stoi for tok in tokens) / len(tokens)

print("OOV rate  train: {:.2%} | validation: {:.2%} | test: {:.2%}".format(
    oov_rate(train_df[TEXT_COL]), oov_rate(val_df[TEXT_COL]), oov_rate(test_df[TEXT_COL])))

# 2) Exact duplicates AFTER normalization, inside and across splits
norm = {
    "train": set(train_df[TEXT_COL].map(clean_text)),
    "validation": set(val_df[TEXT_COL].map(clean_text)),
    "test": set(test_df[TEXT_COL].map(clean_text)),
}
print("\nDuplicates inside train after cleaning:",
      len(train_df) - len(norm["train"]))
print("Shared train/validation:", len(norm["train"] & norm["validation"]))
print("Shared train/test      :", len(norm["train"] & norm["test"]))
print("Shared validation/test :", len(norm["validation"] & norm["test"]))

In [ ]:
# 3) Near-duplicates: how similar is each validation/test text to its closest TRAIN text?
sim_vectorizer = TfidfVectorizer(tokenizer=tokenize, token_pattern=None, lowercase=False)
X_train_sim = sim_vectorizer.fit_transform(train_df[TEXT_COL])   # fitted on train only

def max_similarity(texts, chunk=500):
    X_other = sim_vectorizer.transform(texts)
    best = []
    for start in range(0, X_other.shape[0], chunk):
        sims = linear_kernel(X_other[start:start + chunk], X_train_sim)
        best.append(sims.max(axis=1))
    return np.concatenate(best)

for name, df in [("validation", val_df), ("test", test_df)]:
    sims = max_similarity(df[TEXT_COL])
    print(f"{name}: similarity >= 0.95 with a train text: {(sims >= 0.95).sum()} | "
          f">= 0.80: {(sims >= 0.80).sum()} | median similarity: {np.median(sims):.2f}")

# 4) Authors / identifying elements
patterns = {
    "email-like": r"[\w\.-]+@[\w\.-]+\.\w+",
    "long digit sequences (7+)": r"\d{7,}",
    "URLs": r"http|www\.",
}
for name, pat in patterns.items():
    total = sum(df[TEXT_COL].str.contains(pat, case=False, regex=True).sum()
                for df in (train_df, val_df, test_df))
    print(f"Texts with {name}: {total}")

### Track A notes (leakage and decisions)

- **Fit on train only:** vocabulary, max length and label classes come from the training split. Validation and test are only transformed.
- **Authors / related examples:** no author, customer or conversation column exists, so they cannot be grouped. We rely on the official train/test split.
- **Exact duplicates across splits:** __ (fill from A4).
- **Near-duplicates:** some validation/test texts are paraphrases of train texts (similarity >= 0.80: __). This is normal in Banking77 because each intent has many paraphrases. We keep them and mention it when reporting scores.
- **Identifying information:** none found in the checks above (fill the counts).
- **Handover to Track B:** `clean_text`, `tokenize`, `TextPreprocessor`, the fitted `preprocessor`, and the arrays `X_train`, `X_val`, `X_test`.